In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

#### **Read & Load Data**

In [2]:
import textwrap

def wrap_text(text, width=90): #preserve_newlines
    # Split the input text into lines based on newline characters
    lines = text.split('\n')

    # Wrap each line individually
    wrapped_lines = [textwrap.fill(line, width=width) for line in lines]

    # Join the wrapped lines back together using newline characters
    wrapped_text = '\n'.join(wrapped_lines)

    return wrapped_text

In [ ]:
import langchain
from langchain.vectorstores.chroma import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader

In [4]:
loader_harrypotter  = PyPDFLoader("../harrypotter_1.pdf")
documnet_harrypotter = loader_harrypotter.load()

In [5]:
print(len(documnet_harrypotter))

250


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=100,
    length_function = len,
    is_separator_regex = False,
)

In [7]:
text_harrypotter = text_splitter.split_documents(documnet_harrypotter)

In [8]:
print(len(text_harrypotter))

1155


In [9]:
print(text_harrypotter[4])

page_content='starts, there was nothing about the cloudy sky outside to suggest that
strange and mysterious things would soon be happening all over the
country. Mr. Dursley hummed as he picked out his most boring tie for
work, and Mrs. Dursley gossiped away happily as she wrestled a screaming
Dudley into his high chair.
None of them noticed a large, tawny owl flutter past the window.
At half past eight, Mr. Dursley picked up his briefcase, pecked Mrs.' metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2001-02-13T16:47:14+00:00', 'subject': 'Harry Potter', 'author': 'J.K. Rowling', 'moddate': '2005-11-26T18:01:39+02:00', 'title': "Harry Potter, Book 1; The Sorcerer's Stone", 'source': '../harrypotter_1.pdf', 'total_pages': 250, 'page': 1, 'page_label': '2'}


#### **Load the Embeddings Model**

In [10]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

#### **Vector DB**

In [11]:
texts = [doc.page_content for doc in text_harrypotter]
len(texts)

1155

In [12]:
db = Chroma.from_texts(
    texts=texts, 
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [13]:
query = "Who is harry potter ?"


db.similarity_search(query, k=5)

[Document(metadata={}, page_content='he is?"\n"Who?"\n"Harry Potter!"\nHarry heard the little girl\'s voice.\n"Oh, Mom, can I go on the train and see him, Mom, eh please...."\n"You\'ve already seen him, Ginny, and the poor boy isn\'t something you\ngoggle at in a zoo. Is he really, Fred? How do you know?"\n"Asked him. Saw his scar. It\'s really there - like lightning."\n"Poor dear - no wonder he was alone, I wondered. He was ever so polite\nwhen he asked how to get onto the platform."'),
 Document(metadata={}, page_content='he is?"\n"Who?"\n"Harry Potter!"\nHarry heard the little girl\'s voice.\n"Oh, Mom, can I go on the train and see him, Mom, eh please...."\n"You\'ve already seen him, Ginny, and the poor boy isn\'t something you\ngoggle at in a zoo. Is he really, Fred? How do you know?"\n"Asked him. Saw his scar. It\'s really there - like lightning."\n"Poor dear - no wonder he was alone, I wondered. He was ever so polite\nwhen he asked how to get onto the platform."'),
 Document(meta

#### **Setup a Retriever**

In [15]:
retriever = db.as_retriever() # can add mmr fetch_k=20, search_type="mmr"

retriever.invoke(query)

[Document(metadata={}, page_content='he is?"\n"Who?"\n"Harry Potter!"\nHarry heard the little girl\'s voice.\n"Oh, Mom, can I go on the train and see him, Mom, eh please...."\n"You\'ve already seen him, Ginny, and the poor boy isn\'t something you\ngoggle at in a zoo. Is he really, Fred? How do you know?"\n"Asked him. Saw his scar. It\'s really there - like lightning."\n"Poor dear - no wonder he was alone, I wondered. He was ever so polite\nwhen he asked how to get onto the platform."'),
 Document(metadata={}, page_content='he is?"\n"Who?"\n"Harry Potter!"\nHarry heard the little girl\'s voice.\n"Oh, Mom, can I go on the train and see him, Mom, eh please...."\n"You\'ve already seen him, Ginny, and the poor boy isn\'t something you\ngoggle at in a zoo. Is he really, Fred? How do you know?"\n"Asked him. Saw his scar. It\'s really there - like lightning."\n"Poor dear - no wonder he was alone, I wondered. He was ever so polite\nwhen he asked how to get onto the platform."'),
 Document(meta

#### **Setup a LLM**

In [19]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.0
)

#### **Chat Chain**

In [42]:
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough

In [17]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

In [20]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [21]:
text_reply = chain.invoke("Who gave Harry his first broomstick?")

print(wrap_text(text_reply))

The context does not specify who gave Harry his first broomstick. It only mentions that
Harry received a Nimbus Two Thousand and that Ron was excited about it.


In [24]:
text_reply = chain.invoke("Why were the Dursleys afraid of being associated with the Potters?")

print(wrap_text(text_reply))

The Dursleys were afraid of being associated with the Potters because they considered the
Potters to be unDursleyish and looked down upon them. Mrs. Dursley pretended she didn't
have a sister, as she feared that if anyone found out about the Potters, it would be a
disgrace and they couldn't bear the thought of what the neighbors would think.


#### **With Rag-Fusion**

In [41]:
from langchain.schema.output_parser import StrOutputParser
from langchain.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.prompts import PromptTemplate

In [27]:
prompt = ChatPromptTemplate(
    input_variables=["original_query"],
    messages=[
        SystemMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=[],
                template="You are a helpful assistant that generates multiple search queries based on a single input query.",
            )
        ),
        HumanMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=["original_query"],
                template="Generate multiple search queries related to: {question} \n OUTPUT (4 queries):",
            )
        ),
    ],
)

prompt

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant that generates multiple search queries based on a single input query.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='Generate multiple search queries related to: {question} \n OUTPUT (4 queries):'), additional_kwargs={})])

In [28]:
original_query = "Who gave Harry his first broomstick?"

In [29]:
generate_queries = (
    prompt | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

In [30]:
generate_queries

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant that generates multiple search queries based on a single input query.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='Generate multiple search queries related to: {question} \n OUTPUT (4 queries):'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001DCB5D2AE40>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001DCB5D786E0>, root_client=<openai.OpenAI object at 0x000001DCB5D2BE60>, root_async_client=<openai.AsyncOpenAI object at 0x000001DCB5D2B3B0>, model_name='gpt-4o-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretSt

In [31]:
from typing import Any, List, Tuple
from langchain.load import dumps, loads

def reciprocal_rank_fusion(results: List[List[Any]], k: int = 60) -> List[Tuple[Any, float]]:
    """
    Perform Reciprocal Rank Fusion (RRF) on a list of ranked result lists.

    Args:
        results: List of lists, where each sublist contains ranked documents.
        k: The RRF constant (default 60).

    Returns:
        List of tuples (document, fused_score), sorted by fused_score descending.
    """
    fused_scores = {}
    for docs in results:
        # Assumes the docs are returned in sorted order of relevance
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc_str), score)
        for doc_str, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    return reranked_results

In [32]:
rag_fusion_chain = generate_queries | retriever.map() | reciprocal_rank_fusion

In [33]:
langchain.debug = True

In [35]:
rag_fusion_chain.input_schema.model_json_schema()

{'properties': {'question': {'title': 'Question', 'type': 'string'}},
 'required': ['question'],
 'title': 'PromptInput',
 'type': 'object'}

In [36]:
rag_fusion_chain.invoke({"question": original_query})

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "question": "Who gave Harry his first broomstick?"
}
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "question": "Who gave Harry his first broomstick?"
}
[chain/end] [chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "System: You are a helpful assistant that generates multiple search queries based on a single input query.\nHuman: Generate multiple search queries related to: Who gave Harry his first broomstick? \n OUTPUT (4 queries):"
  ]
}
[llm/end] [chain:RunnableSequence > llm:ChatOpenAI] [1.73s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "1. Who gifted Harry Potter his first broomstick in the series?\n2. What is the story behind Harry's first broomstick gift?\n3. Which ch

C:\Users\Bapan Bairagya\AppData\Local\Temp\ipykernel_9560\1981392677.py:25: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  (loads(doc_str), score)


[(Document(metadata={}, page_content='looking at wands," said the boy. He had a bored, drawling voice. "Then\nI\'m going to drag them off to took at racing brooms. I don\'t see why\nfirst years can\'t have their own. I think I\'ll bully father into getting\nme one and I\'ll smuggle it in somehow."\nHarry was strongly reminded of Dudley.\n"Have you got your own broom?" the boy went on.\n"No," said Harry.\n"Play Quidditch at all?"\n"No," Harry said again, wondering what on earth Quidditch could be.'),
  0.098378283095364),
 (Document(metadata={}, page_content='Professor McGonagall\nHarry had difficulty hiding his glee as he handed the note to Ron to\nread.\n"A Nimbus Two Thousand!" Ron moaned enviously. "I\'ve never even touched\none."\nThey left the hall quickly, wanting to unwrap the broomstick in private\nbefore their first class, but halfway across the entrance hall they\nfound the way upstairs barred by Crabbe and Goyle. Malfoy seized the\npackage from Harry and felt it.\n"That\'s a

In [37]:
from langchain.schema.runnable import RunnablePassthrough

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

full_rag_fusion_chain = (
    {
        "context": rag_fusion_chain,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [38]:
full_rag_fusion_chain

{
  context: ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant that generates multiple search queries based on a single input query.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='Generate multiple search queries related to: {question} \n OUTPUT (4 queries):'), additional_kwargs={})])
           | ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001DCB5D2AE40>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001DCB5D786E0>, root_client=<openai.OpenAI object at 0x000001DCB5D2BE60>, root_async_client=<openai.AsyncOpenAI object at 0x000001DCB5D2B3B0>, model_name='gpt-4o-mini', temperature=0.0, model_kwargs={},

In [39]:
full_rag_fusion_chain.input_schema.model_json_schema()

{'properties': {'question': {'title': 'Question', 'type': 'string'},
  'root': {'title': 'Root'}},
 'required': ['question', 'root'],
 'title': 'RunnableParallel<context,question>Input',
 'type': 'object'}

In [40]:
query = "Who gave Harry his first broomstick?"

response = full_rag_fusion_chain.invoke({"question": query})
response

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "question": "Who gave Harry his first broomstick?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question>] Entering Chain run with input:
{
  "question": "Who gave Harry his first broomstick?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence] Entering Chain run with input:
{
  "question": "Who gave Harry his first broomstick?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "question": "Who gave Harry his first broomstick?"
}
[chain/end] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence > 

'Professor McGonagall gave Harry his first broomstick, which was a Nimbus Two Thousand.'

In [45]:
query = "Why does Uncle Vernon go to extreme lengths to prevent Harry from reading his letters?"

response = full_rag_fusion_chain.invoke({"question": query})
response

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "question": "Why does Uncle Vernon go to extreme lengths to prevent Harry from reading his letters?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question>] Entering Chain run with input:
{
  "question": "Why does Uncle Vernon go to extreme lengths to prevent Harry from reading his letters?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence] Entering Chain run with input:
{
  "question": "Why does Uncle Vernon go to extreme lengths to prevent Harry from reading his letters?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "question": "Why does Uncle Vernon go to extreme lengths to prevent Harry from reading his letters?"
}
[chain/end] [chain:RunnableSequence > chain:RunnableParallel<context,question> > cha

"Uncle Vernon goes to extreme lengths to prevent Harry from reading his letters because he is terrified of the content of the letters, which he believes could reveal something significant about Harry's background or identity. His reaction to the letters, including his face changing colors and his frantic behavior, indicates that he is deeply concerned about what the letters might mean for Harry and their family."

In [46]:
response

"Uncle Vernon goes to extreme lengths to prevent Harry from reading his letters because he is terrified of the content of the letters, which he believes could reveal something significant about Harry's background or identity. His reaction to the letters, including his face changing colors and his frantic behavior, indicates that he is deeply concerned about what the letters might mean for Harry and their family."